# qwen25-7b-medmcqa-kfp-ft-eval-pipeline

[![Open in JupyterLab](https://img.shields.io/badge/Open%20in-JupyterLab-F37626?logo=jupyter&logoColor=white)](http://localhost:8888/lab/tree/git-miramar-labs-org/projects/qwen25-7b-medmcqa-kfp-ft-eval-pipeline/notebook.ipynb)

| | |
|---|---|
| **Type** | KFP v2 eval-first fine-tuning pipeline |
| **Model** | [Qwen/Qwen2.5-7B-Instruct](https://huggingface.co/Qwen/Qwen2.5-7B-Instruct) |
| **Dataset** | [openlifescienceai/medmcqa](https://huggingface.co/datasets/openlifescienceai/medmcqa) |

KFP v2 eval-first fine-tuning pipeline on the Miramar platform.

**Development workflow:**
1. Edit `config.yaml` — set model ID, datasets, thresholds, and judge prompt
2. Edit `formatters.py` — add one formatter function per dataset, register in `FORMATTERS`
3. Write step logic in the `@dsl.component` cells below
4. Save (`Ctrl+S`), run the **Build → `pipeline.py`** cell
5. Trigger **Deploy to KFP** workflow (or run **Compile & Submit** below)

**Pipeline DAG:**
```
prepare_dataset ──► baseline_eval ──► fine_tune ─┬─► post_finetune_eval ──►─┐
                                                 │                          │
                                                 └─► safety_eval ──────────►┤
                                                                            │
                                                                   deployment_gate
```

> `fine_tune` runs after `baseline_eval` (not parallel) — on single-node k3s, both steps
> need GPU memory simultaneously and will exceed the allocatable limit if run together.


## Step Development

Write step logic inside each `@dsl.component` function body.

- **All imports must be inside the function body** — each component runs in its own container
- Set `base_image` and `packages_to_install` to match the step's runtime requirements
- Use `Input[T]` / `Output[T]` to pass artifacts between steps
- GPU steps: call `.set_accelerator_type("nvidia.com/gpu").set_accelerator_limit(1).set_memory_limit("64G")` in the pipeline cell

In [ ]:
from kfp import dsl
from kfp.dsl import Input, Output, Dataset, Model, Metrics, Artifact

### download_model

Downloads the base model to the HF cache PVC before any eval or training step.
Runs in parallel with `prepare_dataset` — CPU-only, no GPU required.

In [ ]:
@dsl.component(
    base_image="python:3.11-slim",
    packages_to_install=["huggingface_hub>=0.21"],
)
def download_model(
    base_model_id: str,
):
    import os
    from huggingface_hub import snapshot_download
    print(f"Downloading {base_model_id} to HF cache...")
    snapshot_download(
        repo_id=base_model_id,
        cache_dir="/root/.cache/huggingface/hub",
        token=os.environ.get("HF_TOKEN"),
    )
    print(f"Download complete: {base_model_id}")

### prepare_dataset

Loads datasets from HuggingFace, applies formatters from `formatters.py`, splits into train/val/test.
The Build cell inlines `formatters.py` at the `# <<< FORMATTERS_INJECT >>>` marker.

In [ ]:
@dsl.component(
    base_image="python:3.11-slim",
    packages_to_install=[
        "datasets<3.0",
        "huggingface_hub>=0.21.2,<0.24",
        "transformers",
    ],
)
def prepare_dataset(
    dataset_names: list,
    val_size: float,
    test_size: float,
    train_out: Output[Dataset],
    val_out: Output[Dataset],
    test_out: Output[Dataset],
    chunk_index: int = 0,
    total_chunks: int = 1,
    chunking_enabled: bool = False,
    shuffle_seed: int = 42,
):
    import json, pathlib, random

    # <<< FORMATTERS_INJECT >>>
    # formatters.py is inlined here by the Build cell. Do not remove this marker.

    # <<< LOADERS_INJECT >>>
    # loaders.py is inlined here by the Build cell. Do not remove this marker.

    missing = [n for n in dataset_names if n not in LOADERS]
    if missing:
        raise ValueError(f"No loader for: {missing}. Add to loaders.py LOADERS dict.")

    all_rows = []
    for name in dataset_names:
        ds = LOADERS[name]()
        all_rows.extend(
            {"instruction": r["instruction"], "response": r["response"], "source": r["source"]}
            for r in ds
        )

    # Fixed seed ensures the same shuffle order across every chunk run —
    # val/test slices stay consistent; only the training slice varies by chunk_index.
    random.seed(shuffle_seed)
    random.shuffle(all_rows)
    n = len(all_rows)
    n_val  = max(1, int(n * val_size))
    n_test = max(1, int(n * test_size))
    train_rows = all_rows[n_val + n_test:]
    val_rows   = all_rows[:n_val]
    test_rows  = all_rows[n_val:n_val + n_test]

    if chunking_enabled and total_chunks > 1:
        chunk_size = len(train_rows) // total_chunks
        start = chunk_index * chunk_size
        end   = start + chunk_size if chunk_index < total_chunks - 1 else len(train_rows)
        train_rows = train_rows[start:end]
        print(f"Chunking: slice [{start}:{end}] of {len(all_rows[n_val + n_test:])} total "
              f"train rows (chunk {chunk_index + 1}/{total_chunks})")

    pathlib.Path(train_out.path).write_text(json.dumps(train_rows))
    pathlib.Path(val_out.path).write_text(json.dumps(val_rows))
    pathlib.Path(test_out.path).write_text(json.dumps(test_rows))
    print(f"Dataset split: {len(train_rows)} train / {len(val_rows)} val / {len(test_rows)} test")

### baseline_eval

Evaluates the base model on the validation set before fine-tuning. Records metrics to MLflow.

In [ ]:
@dsl.component(
    base_image="nvcr.io/nvidia/pytorch:26.04-py3",
    packages_to_install=[
        "transformers>=4.45,<5.0",
        "accelerate",
        "mlflow",
        "nvtx",
    ],
)
def baseline_eval(
    val: Input[Dataset],
    base_model_id: str,
    eval_sample_size: int,
    run_id: str,
    mlflow_tracking_uri: str,
    mlflow_experiment_name: str,
    metrics: Output[Metrics],
    max_new_tokens: int = 1024,
    system_message: str = "You are a helpful assistant.",
    do_sample: bool = False,
):
    import json, pathlib, time, mlflow, nvtx, torch
    from transformers import AutoTokenizer, AutoModelForCausalLM

    val_data = json.loads(pathlib.Path(val.path).read_text())[:eval_sample_size]

    # <<< UTILS_INJECT >>>

    # <<< EVAL_HELPERS_INJECT >>>

    tokenizer = AutoTokenizer.from_pretrained(base_model_id)
    model = AutoModelForCausalLM.from_pretrained(
        base_model_id, dtype=torch.bfloat16, device_map="auto", max_memory={0: "100GiB"})
    model.eval()
    _infer = make_infer_fn(tokenizer, model, system_message, max_new_tokens, do_sample)

    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment(mlflow_experiment_name)
    with mlflow.start_run(run_name=f"{run_id}-baseline"):
        mlflow.log_param("eval_sample_size", len(val_data))

        correct = 0
        with nvtx.annotate("baseline_eval_inference"):
            for row in val_data:
                generated = _infer(row)
                # ---- USER CODE BLOCK ----
                if extract_answer(generated) == extract_answer(row["response"]):  # noqa
                    correct += 1
                # ---- END USER CODE BLOCK ----
            torch.cuda.synchronize()
        accuracy = correct / len(val_data) if val_data else 0.0
        mlflow.log_metric("baseline_accuracy", accuracy)

    pathlib.Path(metrics.path).write_text(json.dumps({"baseline_accuracy": accuracy}))
    print(f"Baseline accuracy: {accuracy:.4f}")

### baseline_safety_eval

In [ ]:
@dsl.component(
    base_image="nvcr.io/nvidia/pytorch:26.04-py3",
    packages_to_install=[
        "transformers>=4.45,<5.0",
        "accelerate",
        "openai",
        "mlflow",
        "nvtx",
    ],
)
def baseline_safety_eval(
    val: Input[Dataset],
    base_model_id: str,
    judge_model_id: str,
    judge_system_prompt: str,
    judge_base_url: str,
    sample_size: int,
    run_id: str,
    mlflow_tracking_uri: str,
    mlflow_experiment_name: str,
    metrics: Output[Metrics],
    max_new_tokens: int = 256,
    system_message: str = "You are a helpful medical assistant.",
    do_sample: bool = False,
):
    import json, pathlib, re, mlflow, nvtx, torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from openai import OpenAI

    val_data = json.loads(pathlib.Path(val.path).read_text())[:sample_size]

    # <<< UTILS_INJECT >>>

    # ---- USER CODE BLOCK ----
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tokenizer = AutoTokenizer.from_pretrained(base_model_id)
    model = AutoModelForCausalLM.from_pretrained(
        base_model_id, torch_dtype=torch.bfloat16, device_map="auto", max_memory={0: "100GiB"})
    model.eval()
    client = OpenAI(base_url=judge_base_url, api_key="ollama")
    scores = []
    for row in val_data:
        messages = [{"role": "user", "content": row["instruction"]}]
        input_ids = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt"
        ).to(model.device)
        with torch.no_grad():
            output_ids = model.generate(
                input_ids, max_new_tokens=max_new_tokens, do_sample=do_sample,
                pad_token_id=tokenizer.eos_token_id)
        generated = tokenizer.decode(output_ids[0][input_ids.shape[-1]:], skip_special_tokens=True)
        try:
            result = client.chat.completions.create(
                model=judge_model_id,
                messages=[
                    {"role": "system", "content": judge_system_prompt},
                    {"role": "user", "content": f"Question: {row['instruction']}\nModel answer: {generated}"},
                ],
                temperature=0.0,
            )
            scores.append(parse_score(result.choices[0].message.content))
        except Exception as e:
            print(f"[baseline_safety_eval] judge error: {e}")
            scores.append(1.0)
    avg_score = sum(scores) / len(scores) if scores else 0.0
    # ---- END USER CODE BLOCK ----

    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment(mlflow_experiment_name)
    with mlflow.start_run(run_name=f"{run_id}-baseline-safety-eval"):
        mlflow.log_metric("baseline_safety_avg_score", avg_score)
        mlflow.log_metric("baseline_safety_sample_size", len(scores))
        mlflow.log_param("judge_model_id", judge_model_id)

    pathlib.Path(metrics.path).write_text(json.dumps({"baseline_safety_avg_score": avg_score}))
    print(f"Baseline safety avg score: {avg_score:.4f} ({len(scores)} samples)")

### fine_tune

Fine-tunes the base model using LoRA on the training set.

In [ ]:
@dsl.component(
    base_image="nvcr.io/nvidia/pytorch:26.04-py3",
    packages_to_install=["wandb>=0.16"],
)
def fine_tune(
    train: Input[Dataset],
    val: Input[Dataset],
    base_model_id: str,
    learning_rate: float,
    num_epochs: int,
    lora_r: int,
    lora_alpha: int,
    lora_dropout: float,
    target_modules: list,
    lora_bias: str,
    lora_task_type: str,
    batch_size: int,
    gradient_accumulation_steps: int,
    max_seq_length: int,
    bf16: bool,
    gradient_checkpointing: bool,
    eval_strategy: str,
    logging_steps: int,
    run_id: str,
    mlflow_tracking_uri: str,
    mlflow_experiment_name: str,
    ft_model: Output[Model],
    target_training_seconds: int = 0,
    pipeline_name: str = "",
    chunk_index: int = 0,
    total_chunks: int = 1,
    wandb_enabled: bool = False,
    wandb_project: str = "",
    wandb_entity: str = "",
    wandb_run_name: str = "",
):
    import subprocess, sys, os
    # NGC containers set PIP_CONSTRAINT=/etc/pip/constraint.txt which pins pyarrow==19.0.1.
    # Clear it so pip can upgrade pyarrow to satisfy trl's datasets dependency.
    env = {**os.environ, "PIP_CONSTRAINT": ""}
    subprocess.run(
        [sys.executable, "-m", "pip", "install",
         "pyarrow>=21.0.0", "transformers>=4.45,<5.0", "peft>=0.15.0", "trl>=0.14.0,<1.0", "accelerate>=0.27.0", "mlflow"],
        check=True, env=env,
    )
    import json, pathlib, shutil, mlflow, time as _time

    train_data = json.loads(pathlib.Path(train.path).read_text())
    val_data   = json.loads(pathlib.Path(val.path).read_text())

    # <<< UTILS_INJECT >>>

    # Adapter paths on the HF cache PVC — deterministic from pipeline_name + chunk_index.
    # PVC is mounted at /root/.cache/huggingface so these persist across pipeline runs.
    _adapter_base = f"/root/.cache/huggingface/adapters/{pipeline_name}" if pipeline_name else ""
    _prev_path = f"{_adapter_base}/chunk-{chunk_index - 1}" if (_adapter_base and chunk_index > 0) else ""
    _out_path  = f"{_adapter_base}/chunk-{chunk_index}" if _adapter_base else ""

    # ---- USER CODE BLOCK ----
    def to_chat(rows):
        from datasets import Dataset as _Dataset
        return _Dataset.from_list([
            {"messages": [
                {"role": "user",      "content": r["instruction"]},
                {"role": "assistant", "content": r["response"]},
            ]}
            for r in rows
        ])
    # ---- END USER CODE BLOCK ----

    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from peft import LoraConfig, get_peft_model, PeftModel
    from trl import SFTTrainer, SFTConfig
    from transformers import TrainerCallback

    tokenizer = AutoTokenizer.from_pretrained(base_model_id)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Warm-start: load previous chunk's adapter from PVC, continue training it.
    # is_trainable=True keeps adapter weights unfrozen without a memory-expensive merge.
    if _prev_path and pathlib.Path(_prev_path).exists():
        print(f"Warm-start: loading adapter from {_prev_path}")
        model = AutoModelForCausalLM.from_pretrained(
            base_model_id, dtype=torch.bfloat16, device_map="auto", max_memory={0: "100GiB"})
        model = PeftModel.from_pretrained(model, _prev_path, is_trainable=True)
        print("Warm-start: adapter loaded as trainable continuation")
    else:
        if _prev_path:
            print(f"Warning: prev adapter path {_prev_path} not found — cold start")
        model = AutoModelForCausalLM.from_pretrained(
            base_model_id, dtype=torch.bfloat16, device_map="auto", max_memory={0: "100GiB"})
        lora_cfg = LoraConfig(r=lora_r, lora_alpha=lora_alpha, lora_dropout=lora_dropout,
            target_modules=target_modules, bias=lora_bias, task_type=lora_task_type)
        model = get_peft_model(model, lora_cfg)

    tokenizer.model_max_length = max_seq_length
    train_ds = to_chat(train_data)
    val_ds   = to_chat(val_data)

    # Self-calibrating step budget: run 10 steps, measure sec/step, then set max_steps.
    # A TrainerCallback stops training at the calibrated step count (control.should_training_stop).
    _report_to = ["mlflow", "wandb"] if wandb_enabled else ["mlflow"]
    if wandb_enabled:
        import wandb as _wandb
        _wandb.init(
            project=wandb_project or mlflow_experiment_name,
            entity=wandb_entity or None,
            name=wandb_run_name or f"{run_id}-finetune",
            config={"learning_rate": learning_rate, "lora_r": lora_r, "lora_alpha": lora_alpha,
                    "batch_size": batch_size, "num_epochs": num_epochs},
        )
    callbacks = []
    if target_training_seconds > 0:
        class _TimeBudgetCallback(TrainerCallback):
            def __init__(self, budget_secs):
                self._budget = budget_secs
                self._t0 = None
                self._max_steps = None

            def on_step_begin(self, args, state, control, **kwargs):
                if state.global_step == 0:
                    self._t0 = _time.perf_counter()
                return control

            def on_step_end(self, args, state, control, **kwargs):
                if self._max_steps is None and state.global_step == 10:
                    _sec = (_time.perf_counter() - self._t0) / 10
                    self._max_steps = max(50, min(int(self._budget / _sec), state.max_steps))
                    state.max_steps = self._max_steps  # update progress display
                    print(f"[TimeBudget] {_sec:.1f} s/step → max_steps={self._max_steps} "
                          f"(budget={self._budget}s)")
                if self._max_steps is not None and state.global_step >= self._max_steps:
                    control.should_training_stop = True
                return control

        callbacks.append(_TimeBudgetCallback(target_training_seconds))
        training_args = SFTConfig(
            output_dir="/tmp/ft_checkpoints", max_steps=999999,
            per_device_train_batch_size=batch_size,
            gradient_accumulation_steps=gradient_accumulation_steps,
            learning_rate=learning_rate, bf16=bf16,
            gradient_checkpointing=gradient_checkpointing,
            eval_strategy="no", logging_steps=logging_steps,
            save_strategy="no", report_to=_report_to, dataloader_num_workers=0)
    else:
        training_args = SFTConfig(
            output_dir="/tmp/ft_checkpoints", num_train_epochs=num_epochs,
            per_device_train_batch_size=batch_size,
            gradient_accumulation_steps=gradient_accumulation_steps,
            learning_rate=learning_rate, bf16=bf16,
            gradient_checkpointing=gradient_checkpointing,
            eval_strategy=eval_strategy, logging_steps=logging_steps,
            save_strategy="no", report_to=_report_to, dataloader_num_workers=0)

    trainer = SFTTrainer(model=model, processing_class=tokenizer, args=training_args,
        train_dataset=train_ds, eval_dataset=val_ds, callbacks=callbacks)

    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment(mlflow_experiment_name)
    with mlflow.start_run(run_name=f"{run_id}-finetune"):
        mlflow.log_params({"learning_rate": learning_rate, "num_epochs": num_epochs,
                           "lora_r": lora_r, "lora_alpha": lora_alpha,
                           "chunk_index": chunk_index,
                           "target_training_seconds": target_training_seconds,
                           "warm_start": bool(_prev_path and pathlib.Path(_prev_path).exists()),
                           "adapter_pvc_path": _out_path})
        trainer.train()
        history = trainer.state.log_history
        train_loss = next((e["loss"]      for e in reversed(history) if "loss"      in e), None)
        eval_loss  = next((e["eval_loss"] for e in reversed(history) if "eval_loss" in e), None)
        if train_loss is not None: mlflow.log_metric("train_loss", train_loss)
        if eval_loss  is not None: mlflow.log_metric("eval_loss",  eval_loss)

    if wandb_enabled:
        try:
            import wandb as _wandb
            _wandb.finish()
        except Exception as _e:
            print(f"[wandb] fine_tune finish failed: {_e}")

    pathlib.Path(ft_model.path).mkdir(parents=True, exist_ok=True)
    model.save_pretrained(ft_model.path)
    tokenizer.save_pretrained(ft_model.path)
    print(f"Fine-tuning complete. Adapter saved to: {ft_model.path}")

    # Copy adapter to persistent PVC path for warm-start in the next chunk run.
    # Done here (not in deployment_gate) so next run always has a valid adapter source
    # regardless of whether this run passes the gate.
    if _out_path:
        try:
            pathlib.Path(_out_path).mkdir(parents=True, exist_ok=True)
            shutil.copytree(ft_model.path, _out_path, dirs_exist_ok=True)
            print(f"Adapter copied to PVC: {_out_path}")
        except Exception as e:
            import sys
            print(f"ERROR: adapter PVC copy failed: {e}", file=sys.stderr)
            raise

### post_finetune_eval

Evaluates the fine-tuned model on the validation set. Results feed into the deployment gate.

In [ ]:
@dsl.component(
    base_image="nvcr.io/nvidia/pytorch:26.04-py3",
    packages_to_install=[
        "transformers>=4.45,<5.0",
        "peft>=0.13",
        "accelerate",
        "mlflow",
        "nvtx",
    ],
)
def post_finetune_eval(
    val: Input[Dataset],
    ft_model: Input[Model],
    base_model_id: str,
    eval_sample_size: int,
    run_id: str,
    mlflow_tracking_uri: str,
    mlflow_experiment_name: str,
    metrics: Output[Metrics],
    max_new_tokens: int = 1024,
    system_message: str = "You are a helpful assistant.",
    do_sample: bool = False,
):
    import json, pathlib, time, mlflow, nvtx, torch

    val_data = json.loads(pathlib.Path(val.path).read_text())[:eval_sample_size]

    # <<< UTILS_INJECT >>>

    # <<< EVAL_HELPERS_INJECT >>>

    # ---- USER CODE BLOCK ----
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from peft import PeftModel
    tokenizer = AutoTokenizer.from_pretrained(ft_model.path)
    model = AutoModelForCausalLM.from_pretrained(
        base_model_id, torch_dtype=torch.bfloat16,
        device_map="auto", max_memory={0: "100GiB"},
    )
    model = PeftModel.from_pretrained(model, ft_model.path)
    model.eval()
    # ---- END USER CODE BLOCK ----
    _infer = make_infer_fn(tokenizer, model, system_message, max_new_tokens, do_sample)

    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment(mlflow_experiment_name)
    with mlflow.start_run(run_name=f"{run_id}-postft-eval"):
        mlflow.log_param("eval_sample_size", len(val_data))

        correct = 0
        with nvtx.annotate("post_finetune_eval_inference"):
            for row in val_data:
                generated = _infer(row)
                # ---- USER CODE BLOCK ----
                if extract_answer(generated) == extract_answer(row["response"]):  # noqa
                    correct += 1
                # ---- END USER CODE BLOCK ----
            torch.cuda.synchronize()
        accuracy = correct / len(val_data) if val_data else 0.0
        mlflow.log_metric("postft_accuracy", accuracy)

    pathlib.Path(metrics.path).write_text(json.dumps({"postft_accuracy": accuracy}))
    print(f"Post-FT accuracy: {accuracy:.4f}")

### safety_eval

LLM-as-judge evaluation of the fine-tuned model's outputs. Uses `OPENAI_API_KEY` injected from the
`mlabs-api-keys` K8s secret. Judge model and system prompt are set in `config.yaml`.

In [ ]:
@dsl.component(
    base_image="nvcr.io/nvidia/pytorch:26.04-py3",
    packages_to_install=[
        "transformers>=4.45,<5.0",
        "peft>=0.13",
        "accelerate",
        "openai",
        "mlflow",
        "nvtx",
    ],
)
def safety_eval(
    val: Input[Dataset],
    ft_model: Input[Model],
    base_model_id: str,
    judge_model_id: str,
    judge_system_prompt: str,
    judge_base_url: str,
    sample_size: int,
    run_id: str,
    mlflow_tracking_uri: str,
    mlflow_experiment_name: str,
    metrics: Output[Metrics],
    max_new_tokens: int = 256,
    system_message: str = "You are a helpful assistant.",
    do_sample: bool = False,
):
    import json, pathlib, mlflow, nvtx, torch
    from openai import OpenAI

    val_data = json.loads(pathlib.Path(val.path).read_text())[:sample_size]

    # <<< UTILS_INJECT >>>

    # parse_score is provided by utils.py via <<< UTILS_INJECT >>> above

    # ---- USER CODE BLOCK ----
    from transformers import AutoTokenizer, AutoModelForCausalLM
    from peft import PeftModel
    tokenizer = AutoTokenizer.from_pretrained(ft_model.path)
    model = AutoModelForCausalLM.from_pretrained(
        base_model_id, torch_dtype=torch.bfloat16, device_map="auto", max_memory={0: "100GiB"})
    model = PeftModel.from_pretrained(model, ft_model.path)
    model.eval()
    client = OpenAI(base_url=judge_base_url, api_key="ollama")
    scores = []
    for row in val_data:
        messages = [{"role": "user", "content": row["instruction"]}]
        input_ids = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, return_tensors="pt"
        ).to(model.device)
        with torch.no_grad():
            with nvtx.annotate("safety_eval_inference"):
                output_ids = model.generate(
                    input_ids, max_new_tokens=max_new_tokens, do_sample=do_sample,
                    pad_token_id=tokenizer.eos_token_id)
        generated = tokenizer.decode(output_ids[0][input_ids.shape[-1]:], skip_special_tokens=True)
        try:
            result = client.chat.completions.create(
                model=judge_model_id,
                messages=[
                    {"role": "system", "content": judge_system_prompt},
                    {"role": "user", "content": f"Question: {row['instruction']}\nModel answer: {generated}"},
                ],
                temperature=0.0,
            )
            scores.append(parse_score(result.choices[0].message.content))
        except Exception as e:
            print(f"[safety_eval] judge error: {e}")
            scores.append(1.0)
    avg_score = sum(scores) / len(scores) if scores else 0.0
    # ---- END USER CODE BLOCK ----

    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment(mlflow_experiment_name)
    with mlflow.start_run(run_name=f"{run_id}-safety-eval"):
        mlflow.log_metric("safety_avg_score", avg_score)

    pathlib.Path(metrics.path).write_text(json.dumps({"safety_avg_score": avg_score}))
    print(f"Safety avg score: {avg_score:.4f}")

### deployment_gate

Compares baseline vs post-FT accuracy and safety score against thresholds from `config.yaml`.
Fails the pipeline if either check does not pass. On pass, writes `gate_result.json` to
the shared PVC for the serving `build-push.yaml` to find.

In [ ]:
@dsl.component(
    base_image="python:3.11-slim",
    packages_to_install=["wandb>=0.16"],
)
def deployment_gate(
    test: Input[Dataset],
    ft_model: Input[Model],
    baseline_metrics: Input[Metrics],
    postft_metrics: Input[Metrics],
    safety_metrics: Input[Metrics],
    baseline_safety_metrics: Input[Metrics],
    accuracy_delta_threshold: float,
    safety_score_threshold: float,
    safety_delta_threshold: float,
    run_id: str,
    pipeline_name: str = "",
    base_model_id: str = "",
    mlflow_experiment_name: str = "",
    wandb_enabled: bool = False,
    wandb_project: str = "",
    wandb_entity: str = "",
    wandb_run_name: str = "",
):
    import datetime, json, pathlib

    baseline = json.loads(pathlib.Path(baseline_metrics.path).read_text())
    postft   = json.loads(pathlib.Path(postft_metrics.path).read_text())
    safety   = json.loads(pathlib.Path(safety_metrics.path).read_text())
    baseline_safety = json.loads(pathlib.Path(baseline_safety_metrics.path).read_text())

    # TODO: update metric keys to match what baseline_eval / post_finetune_eval log
    baseline_acc = baseline.get("baseline_accuracy", 0.0)
    postft_acc   = postft.get("postft_accuracy", 0.0)
    safety_score = safety.get("safety_avg_score", 0.0)
    baseline_safety_score = baseline_safety.get("baseline_safety_avg_score", 0.0)

    acc_delta = postft_acc - baseline_acc
    safety_delta = safety_score - baseline_safety_score
    passed = (acc_delta >= accuracy_delta_threshold
              and safety_delta >= safety_delta_threshold
              and safety_score >= safety_score_threshold)

    print(f"Accuracy delta : {acc_delta:+.4f}  (threshold ≥ {accuracy_delta_threshold:.4f})")
    print(f"Safety delta   : {safety_delta:+.4f}  (threshold ≥ {safety_delta_threshold:.4f})")
    print(f"Safety score   : {safety_score:.4f}  (floor ≥ {safety_score_threshold:.4f})")
    print(f"Gate           : {'PASS' if passed else 'FAIL'}")

    if wandb_enabled:
        try:
            import wandb as _wandb
            _wandb.init(
                project=wandb_project or mlflow_experiment_name,
                entity=wandb_entity or None,
                name=wandb_run_name or f"{run_id}-gate",
            )
            _wandb.log({
                "baseline_accuracy": baseline_acc,
                "postft_accuracy": postft_acc,
                "accuracy_delta": acc_delta,
                "baseline_safety_score": baseline_safety_score,
                "safety_score": safety_score,
                "gate_pass": int(passed),
            })
            _wandb.finish()
        except Exception as _e:
            print(f"[wandb] gate summary failed: {_e}")

    if not passed:
        raise RuntimeError(
            f"Deployment gate failed — acc delta {acc_delta:+.4f}, safety delta {safety_delta:+.4f}, safety score {safety_score:.4f}"
        )

    result = {
        "project": pipeline_name,
        "run_id": run_id,
        "base_model_id": base_model_id,
        "eval_passed": bool(passed),
        "safety_passed": bool(safety_score >= safety_score_threshold),
        "acc_delta": round(acc_delta, 6),
        "safety_score": round(safety_score, 6),
        "adapter_rel_path": f"adapters/{pipeline_name}/chunk-0",
        "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
    }
    out_dir = pathlib.Path(f"/root/.cache/huggingface/runs/{pipeline_name}/{run_id}")
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "gate_result.json").write_text(json.dumps(result, indent=2))
    print(f"Gate result written: {out_dir}/gate_result.json")

### Pipeline

Wire the steps together. Defaults are read from `config.yaml` at import time.
To override a default when submitting a run, pass it as an argument to `create_run_from_pipeline_package`
or set it in the KFP UI.

In [ ]:
import yaml as _yaml, pathlib as _pathlib

_cfg = _yaml.safe_load(_pathlib.Path("config.yaml").read_text())
_dataset_names = [d["name"] for d in _cfg["datasets"]]

# Chunking + time-budget constants — baked into pipeline at build time (not pipeline params).
# Only chunk_index is a pipeline param so it can be overridden per-run at deploy time.
_chunking = _cfg.get("chunking", {})
_target_hours   = _cfg["training"].get("target_hours", 0)
_overhead_hours = _cfg["training"].get("overhead_hours", 0)
_target_training_seconds = int((_target_hours - _overhead_hours) * 3600) if _target_hours > 0 else 0
_wandb_cfg = _cfg.get("wandb", {})
_pipeline_name  = "qwen25-7b-medmcqa-kfp-ft-eval-pipeline"  # must match @dsl.pipeline(name=...)

from kfp import kubernetes as k8s_ext


@dsl.pipeline(name="qwen25-7b-medmcqa-kfp-ft-eval-pipeline")
def pipeline(
    base_model_id: str = _cfg["model"]["id"],
    judge_model_id: str = _cfg["judge"]["model"],
    judge_system_prompt: str = _cfg["judge"]["system_prompt"],
    judge_base_url: str = _cfg["judge"]["base_url"],
    dataset_names: list = _dataset_names,
    learning_rate: float = _cfg["training"]["learning_rate"],
    num_epochs: int = _cfg["training"]["num_epochs"],
    lora_r: int = _cfg["lora"]["r"],
    lora_alpha: int = _cfg["lora"]["alpha"],
    lora_dropout: float = _cfg["lora"]["dropout"],
    target_modules: list = _cfg["lora"]["target_modules"],
    lora_bias: str = _cfg["lora"]["bias"],
    lora_task_type: str = _cfg["lora"]["task_type"],
    batch_size: int = _cfg["training"]["batch_size"],
    gradient_accumulation_steps: int = _cfg["training"]["gradient_accumulation_steps"],
    max_seq_length: int = _cfg["training"]["max_seq_length"],
    bf16: bool = _cfg["training"]["bf16"],
    gradient_checkpointing: bool = _cfg["training"]["gradient_checkpointing"],
    eval_strategy: str = _cfg["training"]["eval_strategy"],
    logging_steps: int = _cfg["training"]["logging_steps"],
    val_size: float = _cfg["training"]["val_size"],
    test_size: float = _cfg["training"]["test_size"],
    eval_sample_size: int = _cfg["eval"]["sample_size"],
    safety_sample_size: int = _cfg["eval"]["safety_sample_size"],
    max_new_tokens: int = _cfg["eval"]["max_new_tokens"],
    safety_max_new_tokens: int = _cfg["eval"]["safety_max_new_tokens"],
    do_sample: bool = _cfg["eval"]["do_sample"],
    system_message: str = _cfg["eval"]["system_message"],
    accuracy_delta_threshold: float = _cfg["eval"]["accuracy_delta_threshold"],
    safety_score_threshold: float = _cfg["eval"]["safety_score_threshold"],
    safety_delta_threshold: float = _cfg["eval"]["safety_delta_threshold"],
    run_id: str = "run-001",
    mlflow_tracking_uri: str = "http://mlflow-tracking.mlflow-system.svc.cluster.local",
    mlflow_experiment_name: str = "default",
    chunk_index: int = _chunking.get("chunk_index", 0),
    wandb_enabled: bool = _wandb_cfg.get("enabled", False),
    wandb_project: str = _wandb_cfg.get("project", ""),
    wandb_entity: str = _wandb_cfg.get("entity", ""),
):
    prep = prepare_dataset(
        dataset_names=dataset_names,
        val_size=val_size,
        test_size=test_size,
        chunk_index=chunk_index,
        total_chunks=_chunking.get("total_chunks", 1),
        chunking_enabled=_chunking.get("enabled", False),
        shuffle_seed=_chunking.get("shuffle_seed", 42),
    )

    dl = download_model(base_model_id=base_model_id)
    prep.after(dl)  # prepare_dataset runs after model is downloaded

    base_eval = baseline_eval(
        val=prep.outputs["val_out"],
        base_model_id=base_model_id,
        eval_sample_size=eval_sample_size,
        run_id=run_id,
        mlflow_tracking_uri=mlflow_tracking_uri,
        mlflow_experiment_name=mlflow_experiment_name,
        max_new_tokens=max_new_tokens,
        system_message=system_message,
        do_sample=do_sample,
    )
    base_eval.set_accelerator_type("nvidia.com/gpu").set_accelerator_limit(1).set_memory_limit("64G")
    base_eval.after(dl)
    # To profile with Nsight Operator: kubernetes.add_pod_label(base_eval, "nvidia-nsight-profile", "enabled")

    baseline_safety = baseline_safety_eval(
        val=prep.outputs["val_out"],
        base_model_id=base_model_id,
        judge_model_id=judge_model_id,
        judge_system_prompt=judge_system_prompt,
        judge_base_url=judge_base_url,
        sample_size=safety_sample_size,
        run_id=run_id,
        mlflow_tracking_uri=mlflow_tracking_uri,
        mlflow_experiment_name=mlflow_experiment_name,
        max_new_tokens=safety_max_new_tokens,
        system_message=system_message,
        do_sample=do_sample,
    )
    baseline_safety.after(base_eval)
    baseline_safety.set_accelerator_type("nvidia.com/gpu").set_accelerator_limit(1).set_memory_limit("64G")

    ft = fine_tune(
        train=prep.outputs["train_out"],
        val=prep.outputs["val_out"],
        base_model_id=base_model_id,
        learning_rate=learning_rate,
        num_epochs=num_epochs,
        lora_r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        target_modules=target_modules,
        lora_bias=lora_bias,
        lora_task_type=lora_task_type,
        batch_size=batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        max_seq_length=max_seq_length,
        bf16=bf16,
        gradient_checkpointing=gradient_checkpointing,
        eval_strategy=eval_strategy,
        logging_steps=logging_steps,
        run_id=run_id,
        mlflow_tracking_uri=mlflow_tracking_uri,
        mlflow_experiment_name=mlflow_experiment_name,
        target_training_seconds=_target_training_seconds,
        pipeline_name=_pipeline_name,
        chunk_index=chunk_index,
        total_chunks=_chunking.get("total_chunks", 1),
        wandb_enabled=wandb_enabled,
        wandb_project=wandb_project,
        wandb_entity=wandb_entity,
        wandb_run_name=run_id,
    )
    # fine_tune runs after baseline_safety_eval — on single-node k3s, both steps need GPU
    # memory simultaneously and will exceed the allocatable limit if scheduled in parallel.
    ft.after(baseline_safety)
    ft.set_accelerator_type("nvidia.com/gpu").set_accelerator_limit(1).set_memory_limit("96G")

    post_eval = post_finetune_eval(
        val=prep.outputs["val_out"],
        ft_model=ft.outputs["ft_model"],
        base_model_id=base_model_id,
        eval_sample_size=eval_sample_size,
        run_id=run_id,
        mlflow_tracking_uri=mlflow_tracking_uri,
        mlflow_experiment_name=mlflow_experiment_name,
        max_new_tokens=max_new_tokens,
        system_message=system_message,
        do_sample=do_sample,
    )
    post_eval.set_accelerator_type("nvidia.com/gpu").set_accelerator_limit(1).set_memory_limit("64G")

    safety = safety_eval(
        val=prep.outputs["val_out"],
        ft_model=ft.outputs["ft_model"],
        base_model_id=base_model_id,
        judge_model_id=judge_model_id,
        judge_system_prompt=judge_system_prompt,
        judge_base_url=judge_base_url,
        sample_size=safety_sample_size,
        run_id=run_id,
        mlflow_tracking_uri=mlflow_tracking_uri,
        mlflow_experiment_name=mlflow_experiment_name,
        max_new_tokens=safety_max_new_tokens,
        system_message=system_message,
        do_sample=do_sample,
    )
    safety.set_accelerator_type("nvidia.com/gpu").set_accelerator_limit(1).set_memory_limit("64G")

    gate = deployment_gate(
        test=prep.outputs["test_out"],
        ft_model=ft.outputs["ft_model"],
        baseline_metrics=base_eval.outputs["metrics"],
        postft_metrics=post_eval.outputs["metrics"],
        safety_metrics=safety.outputs["metrics"],
        baseline_safety_metrics=baseline_safety.outputs["metrics"],
        accuracy_delta_threshold=accuracy_delta_threshold,
        safety_score_threshold=safety_score_threshold,
        safety_delta_threshold=safety_delta_threshold,
        pipeline_name=_pipeline_name,
        base_model_id=base_model_id,
        run_id=run_id,
        mlflow_experiment_name=mlflow_experiment_name,
        wandb_enabled=wandb_enabled,
        wandb_project=wandb_project,
        wandb_entity=wandb_entity,
        wandb_run_name=run_id,
    )

    _SECRET = "mlabs-api-keys"
    _SECRET_KEYS = [
        "OPENAI_API_KEY", "HF_TOKEN", "ANTHROPIC_API_KEY",
        "WANDB_API_KEY", "LANGCHAIN_API_KEY", "NGC_API_KEY", "NVIDIA_API_KEY",
    ]
    _key_map = {k: k for k in _SECRET_KEYS}
    _HF_CACHE_PVC = "hf-model-cache"
    for _task in [dl, base_eval, baseline_safety, ft, post_eval, safety, gate]:
        k8s_ext.mount_pvc(_task, pvc_name=_HF_CACHE_PVC, mount_path="/root/.cache/huggingface")

    for _task in [prep, dl, base_eval, baseline_safety, ft, post_eval, safety, gate]:
        k8s_ext.use_secret_as_env(_task, _SECRET, _key_map)

## Build → `pipeline.py`

Save the notebook first (`Ctrl+S`), then run this cell.

What this does:
1. Reads `formatters.py` and inlines it into the `prepare_dataset` component body
2. Copies all other step cells unchanged
3. Copies the pipeline cell (which reads `config.yaml` at import time)
4. Writes `pipeline.py`

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "scripts/build_pipeline.py"], check=True)

## Compile & Submit

Run these cells to compile and submit directly from Jupyter (requires SSH tunnel).

In [ ]:
from kfp import compiler
from pipeline import pipeline

compiler.Compiler().compile(pipeline_func=pipeline, package_path="/tmp/pipeline.yaml")
print("Compiled: /tmp/pipeline.yaml")

In [ ]:
# Requires SSH tunnel: ssh -L 8080:localhost:8080 <user>@spark-79b7.local
import kfp

client = kfp.Client(host="http://localhost:8080")

In [ ]:
run = client.create_run_from_pipeline_package(
    pipeline_file="/tmp/pipeline.yaml",
    arguments={},
    run_name="notebook-run",
)
print(f"Run ID: {run.run_id}")

In [ ]:
import time

run_id = run.run_id  # or paste a run ID here
for _ in range(20):
    r = client.get_run(run_id)
    state = r.state
    print(f"{state}")
    if state in ("SUCCEEDED", "FAILED", "CANCELED", "SKIPPED"):
        break
    time.sleep(30)